# S&P 500 — Pipeline de Construção da Base de Dados
## TCC: Pair Trading

---

### Fluxo geral

```
ETAPA 1  │ Importações e configuração de caminhos
ETAPA 2  │ Carregar a base original do professor (1990/S2 – 2015/S1)
ETAPA 3  │ Carregar o Periods.csv e construir o mapeamento semestral
ETAPA 4  │ Unir base de preços + mapeamento → sp500_with_periods
ETAPA 5  │ Carregar e validar os constituintes históricos do S&P 500
ETAPA 6  │ Gerar snapshots semestrais e lista de tickers
ETAPA 7  │ Validação do tipo de ajuste de preço (Close vs Adj Close)
ETAPA 8  │ Coletar preços via Yahoo Finance
ETAPA 9  │ Validação tripla Base × Yahoo × Tiingo
ETAPA 10 │ Coletar tickers delisted via Tiingo (Grupo A: rename, Grupo B: Tiingo)
ETAPA 10B│ Recuperar tickers ainda faltantes (falhas de rename + ticker alternativos)
ETAPA 11 │ Unir tudo: base do professor + Yahoo + Tiingo → base final
           (com correção da transição e diagnóstico de escala)
```

> **Nota sobre escala na junção 2015:** empresas com splits ocorridos
> APÓS 2015 terão um salto de preço na junção entre as duas bases.
> Isso é esperado e **não afeta pair trading**, que opera sobre
> retornos logarítmicos, não sobre níveis de preço absolutos.


---
## Etapa 1 — Importações e configuração de caminhos

Centralizar todos os caminhos de arquivo aqui facilita mover o projeto entre
máquinas: basta ajustar `BASE_DIR`.


In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import time
from pathlib import Path

print(f"pandas   {pd.__version__}")
print(f"yfinance {yf.__version__}")

# Nota sobre versões do yfinance:
# Em versões recentes (≥ 0.2.x), ao usar auto_adjust=False, o retorno inclui
# as colunas: Open, High, Low, Close, Adj Close, Volume
# A coluna 'Close' é o preço sem ajuste de dividendos (split-adjusted apenas)
# A coluna 'Adj Close' inclui ajuste de dividendos — NÃO usar para esta base


pandas   2.2.3
yfinance 1.2.0


In [2]:
# ── Caminho raiz do projeto ───────────────────────────────────────────────────
BASE_DIR        = Path(r"C:\Users\jvlei\Desktop\TCC-pair-trading\data_bases")
BASE_DIR_OUTPUT = BASE_DIR / "output"

# ── Arquivos de entrada ───────────────────────────────────────────────────────
PATH_DB       = BASE_DIR / "Pt_formatted.csv"
PATH_DB_FULL  = BASE_DIR / "Pt_formatted.csv"
PATH_PERIODS  = BASE_DIR / "Periods.csv"
PATH_SP500HIS = BASE_DIR / "S&P 500 Historical Components & Changes(01-17-2026).csv"

# ── Arquivos de saída ─────────────────────────────────────────────────────────
PATH_DB_PERIODS       = BASE_DIR_OUTPUT / "sp500_with_periods.csv"
PATH_SNAPSHOTS        = BASE_DIR_OUTPUT / "snapshots_semestrais.csv"
PATH_TICKERS          = BASE_DIR_OUTPUT / "tickers_a_coletar.csv"
PATH_REPORT           = BASE_DIR_OUTPUT / "coleta_report.csv"
PATH_DELISTED_REPORT  = BASE_DIR_OUTPUT / "delisted_report.csv"
PATH_RECOVERY_REPORT  = BASE_DIR_OUTPUT / "recovery_report.csv"   # Etapa 10B
PATH_FINAL_DB         = BASE_DIR_OUTPUT / "sp500_final.csv"
DIR_PRICES            = BASE_DIR_OUTPUT / "prices"
DIR_PRICES_TIINGO     = BASE_DIR_OUTPUT / "prices_tiingo"

DIR_PRICES.mkdir(parents=True, exist_ok=True)
DIR_PRICES_TIINGO.mkdir(parents=True, exist_ok=True)

print("Caminhos configurados:")
for name, p in [
    ("Base do professor", PATH_DB),
    ("Periods.csv",       PATH_PERIODS),
    ("Constituintes S&P", PATH_SP500HIS),
]:
    status = "✅ existe" if p.exists() else "❌ NÃO encontrado"
    print(f"  {status}  {p.name}")


Caminhos configurados:
  ✅ existe  Pt_formatted.csv
  ✅ existe  Periods.csv
  ✅ existe  S&P 500 Historical Components & Changes(01-17-2026).csv


---
## Etapa 2 — Base original do professor (1990/S2 – 2015/S1)

Formato wide: cada linha é um dia de pregão, cada coluna é um ticker.
Os valores são preços de fechamento ajustados por splits.
Empresas que faliram ou foram removidas do índice mantêm seus dados
históricos — identificáveis pelo sufixo `Q` no ticker (ex: `ENRNQ` = Enron).


In [3]:
db = pd.read_csv(PATH_DB)

print(f"Shape: {db.shape[0]:,} linhas × {db.shape[1]:,} colunas")
print(f"Primeiros tickers: {db.columns[:8].tolist()}")
print(f"Últimos  tickers: {db.columns[-5:].tolist()}")


Shape: 6,426 linhas × 1,100 colunas
Primeiros tickers: ['UN', '1005945D', '162007Q', 'RDPL', 'BGG', 'SKY', 'GOSHA', 'HSY']
Últimos  tickers: ['EQIX', '1436513D', '1431816D', 'SPGI', '1448062D']


In [4]:
# Inspeção rápida: empresas famosas para verificar que os valores fazem sentido.
# Todos os valores são split-adjusted, então AAPL em 1990 aparece como ~$1.
spot_check = {
    "AAPL": "Apple — ~$1 em 1990 (split-adjusted)",
    "GE":   "GE — presente desde o início",
    "XOM":  "ExxonMobil",
}
for ticker, nota in spot_check.items():
    if ticker in db.columns:
        primeiros = db[ticker].dropna().head(3).round(4).tolist()
        print(f"{ticker:6s} ({nota}): {primeiros}")


AAPL   (Apple — ~$1 em 1990 (split-adjusted)): [1.3753, 1.3753, 1.3596]
GE     (GE — presente desde o início): [2.8747, 2.9053, 2.8645]
XOM    (ExxonMobil): [5.6624, 5.6328, 5.5438]


---
## Etapa 3 — Mapeamento de períodos (Periods.csv)

`Periods.csv` contém a contagem oficial de dias de pregão da NYSE por semestre,
fornecida pelo professor. Essa contagem é usada como **fonte verdadeira** para
mapear cada linha da base a um período — não usamos aproximações de calendário.

**Estrutura dos 51 semestres:**
- `idx 0`  → `1990/S2` (jul–dez 1990) ← a base começa no 2º semestre de 1990
- `idx 1`  → `1991/S1` (jan–jun 1991)
- `idx 2`  → `1991/S2` ...
- `idx 50` → `2015/S1` (jan–jun 2015) ← último semestre da base do professor

**Âncoras de validação:**
- `2001/S2` tem **123 dias** (único semestre <124) → reflexo do fechamento da NYSE após o 11/set
- `GOOGL` aparece pela primeira vez na linha **3.563** → cai em `2004/S2` (IPO ago/2004)


In [5]:
# Carregar sem cabeçalho (a primeira linha já é dado)
periods_raw = pd.read_csv(PATH_PERIODS, header=None)
periods_raw.columns = ["dias_sem", "nan1", "nan2", "dias_ano"]
periods_raw = periods_raw[["dias_sem", "dias_ano"]]

print(f"Semestres encontrados: {len(periods_raw)}")
print(f"Total de dias de pregão: {periods_raw['dias_sem'].sum():,}")
print()
print(periods_raw)


Semestres encontrados: 51
Total de dias de pregão: 6,425

    dias_sem  dias_ano
0        127     252.0
1        125     253.0
2        128     254.0
3        126     254.0
4        128     252.0
5        124     252.0
6        128     253.0
7        125     252.0
8        127     253.0
9        126     252.0
10       126     252.0
11       126     254.0
12       128     253.0
13       125     253.0
14       128     252.0
15       124     252.0
16       128     252.0
17       124     252.0
18       128     254.0
19       126     252.0
20       126     251.0
21       125     248.0
22       123     247.0
23       124     252.0
24       128     252.0
25       124     252.0
26       128     252.0
27       124     252.0
28       128     253.0
29       125     252.0
30       127     252.0
31       125     251.0
32       126     250.0
33       124     251.0
34       127     252.0
35       125     253.0
36       128     252.0
37       124     252.0
38       128     252.0
39       124     252.0

In [6]:
# Construir mapeamento linha → semestre usando Periods.csv como fonte verdadeira.
# Regra de indexação:
#   idx 0          → 1990/S2 (caso especial)
#   idx ímpar ≥ 1  → S1 do ano (1990 + idx // 2)
#   idx par   ≥ 2  → S2 do ano (1990 + idx // 2)

registros = []
linha_atual = 0

for i, row in periods_raw.iterrows():
    n_dias = int(row["dias_sem"])

    if i == 0:
        ano, sem = 1990, 2
        inicio_sem, fim_sem = "1990-07-01", "1990-12-31"
    else:
        ano = 1990 + (i // 2)
        if i % 2 == 1:      # ímpar → primeiro semestre
            sem = 1
            inicio_sem = f"{ano}-01-01"
            fim_sem    = f"{ano}-06-30"
        else:               # par → segundo semestre
            sem = 2
            inicio_sem = f"{ano}-07-01"
            fim_sem    = f"{ano}-12-31"

    for d in range(n_dias):
        registros.append({
            "linha":           linha_atual,
            "ano":             ano,
            "semestre":        sem,
            "periodo":         f"{ano}/S{sem}",
            "inicio_semestre": inicio_sem,
            "fim_semestre":    fim_sem,
            "dia_no_semestre": d + 1,
            "total_dias_sem":  n_dias,
            "total_dias_ano":  row["dias_ano"],
        })
        linha_atual += 1

mapa = pd.DataFrame(registros)
print(f"Linhas mapeadas: {len(mapa):,}")
print()
print("Início:")
print(mapa.head(3)[["linha", "periodo", "dia_no_semestre", "total_dias_sem"]])
print("\nFim:")
print(mapa.tail(3)[["linha", "periodo", "dia_no_semestre", "total_dias_sem"]])


Linhas mapeadas: 6,425

Início:
   linha  periodo  dia_no_semestre  total_dias_sem
0      0  1990/S2                1             127
1      1  1990/S2                2             127
2      2  1990/S2                3             127

Fim:
      linha  periodo  dia_no_semestre  total_dias_sem
6422   6422  2015/S2              125             127
6423   6423  2015/S2              126             127
6424   6424  2015/S2              127             127


In [7]:
# ── Validações do mapeamento ──────────────────────────────────────────────────

# Validação 1: 2001/S2 deve ter exatamente 123 dias (11 de setembro)
dias_2001s2 = len(mapa[mapa["periodo"] == "2001/S2"])
ok1 = dias_2001s2 == 123
print(f"{'✅' if ok1 else '❌'} 2001/S2 tem {dias_2001s2} dias (esperado: 123 — efeito 11/set)")

# Validação 2: GOOGL deve aparecer em 2004/S2 (IPO agosto/2004)
if "GOOGL" in db.columns:
    idx_googl = db["GOOGL"].first_valid_index()
    periodo_googl = mapa.iloc[idx_googl]["periodo"]
    dia_googl = mapa.iloc[idx_googl]["dia_no_semestre"]
    ok2 = periodo_googl == "2004/S2"
    print(f"{'✅' if ok2 else '❌'} GOOGL aparece na linha {idx_googl} → {periodo_googl}, "
          f"dia {dia_googl} do semestre (esperado: 2004/S2)")

# Validação 3: diferença de linhas entre db e mapa
diff = abs(len(db) - len(mapa))
ok3 = diff <= 1
print(f"{'✅' if ok3 else '⚠️ '} db tem {len(db):,} linhas, mapa tem {len(mapa):,} "
      f"(diferença: {diff} — ok se ≤ 1)")


✅ 2001/S2 tem 123 dias (esperado: 123 — efeito 11/set)
✅ GOOGL aparece na linha 3563 → 2004/S2, dia 35 do semestre (esperado: 2004/S2)
✅ db tem 6,426 linhas, mapa tem 6,425 (diferença: 1 — ok se ≤ 1)


---
## Etapa 4 — Unir base de preços com metadados de período

Adiciona 8 colunas de contexto temporal no início da base:
`ano`, `semestre`, `periodo`, `inicio_semestre`, `fim_semestre`,
`dia_no_semestre`, `total_dias_sem`, `total_dias_ano`.

O resultado é salvo em `sp500_with_periods.csv`.


In [8]:
# Ajustar tamanho do mapa para coincidir com db (diferença de ±1 é normal)
if len(mapa) > len(db):
    mapa_alinhado = mapa.iloc[:len(db)].reset_index(drop=True)
elif len(mapa) < len(db):
    extras = pd.DataFrame([{
        "linha": len(mapa) + k, "ano": np.nan, "semestre": np.nan,
        "periodo": np.nan, "inicio_semestre": np.nan, "fim_semestre": np.nan,
        "dia_no_semestre": np.nan, "total_dias_sem": np.nan, "total_dias_ano": np.nan,
    } for k in range(len(db) - len(mapa))])
    mapa_alinhado = pd.concat([mapa, extras], ignore_index=True)
else:
    mapa_alinhado = mapa.reset_index(drop=True)

cols_periodo = [
    "ano", "semestre", "periodo",
    "inicio_semestre", "fim_semestre",
    "dia_no_semestre", "total_dias_sem", "total_dias_ano",
]

sp500_with_periods = pd.concat(
    [mapa_alinhado[cols_periodo], db.reset_index(drop=True)],
    axis=1
)

print(f"Shape final: {sp500_with_periods.shape[0]:,} linhas × {sp500_with_periods.shape[1]:,} colunas")
print(f"Primeiro período: {sp500_with_periods['periodo'].iloc[0]}")
print(f"Último  período: {sp500_with_periods['periodo'].dropna().iloc[-1]}")


Shape final: 6,426 linhas × 1,108 colunas
Primeiro período: 1990/S2
Último  período: 2015/S2


In [9]:
# Salvar a base unificada
sp500_with_periods.to_csv(PATH_DB_PERIODS, index=False)
print(f"✅ Salvo em: {PATH_DB_PERIODS.name}")
print()

# Amostra para inspeção visual
sp500_with_periods[["periodo", "ano", "semestre", "dia_no_semestre", "AAPL", "GE", "XOM"]].head(5)


✅ Salvo em: sp500_with_periods.csv



,periodo,ano,semestre,dia_no_semestre,AAPL,GE,XOM
0,1990/S2,1990.0,2.0,1.0,1.3753,2.8747,5.6624
1,1990/S2,1990.0,2.0,2.0,1.3753,2.9053,5.6328
2,1990/S2,1990.0,2.0,3.0,1.3596,2.8645,5.5438
3,1990/S2,1990.0,2.0,4.0,1.3987,2.8798,5.6624
4,1990/S2,1990.0,2.0,5.0,1.4573,2.9104,5.6180


---
## Etapa 5 — Constituintes históricos do S&P 500 (1996–2026)

O arquivo `S&P 500 Historical Components & Changes` (repositório `fja05680/sp500`)
registra cada mudança de composição do índice desde 1996. Cada linha representa
uma data em que a composição mudou, com a lista completa de tickers naquele momento.

Para saber quem estava no índice em qualquer data D: buscamos a última linha
com `date ≤ D`.

**Validações usadas:**
- Tesla (`TSLA`): entrou em **21/dez/2020** (não antes)
- Facebook/Meta: era `FB` até **08/jun/2022**, virou `META` em 09/jun/2022
- Lehman Brothers (`LEHMQ`): removida após a falência em setembro de 2008


In [10]:
hist = pd.read_csv(PATH_SP500HIS)
hist["date"] = pd.to_datetime(hist["date"])
hist["n_tickers"] = hist["tickers"].apply(lambda x: len(x.split(",")))

print(f"Snapshots carregados: {len(hist):,}")
print(f"Período coberto: {hist['date'].min().date()} → {hist['date'].max().date()}")
print(f"Tickers por snapshot: min={hist['n_tickers'].min()}, max={hist['n_tickers'].max()}, "
      f"média={hist['n_tickers'].mean():.0f}")


Snapshots carregados: 2,705
Período coberto: 1996-01-02 → 2026-01-14
Tickers por snapshot: min=487, max=507, média=497


In [11]:
def get_constituents(date_str):
    """Retorna o conjunto de tickers do S&P 500 em uma data específica."""
    date = pd.to_datetime(date_str)
    subset = hist[hist["date"] <= date]
    if len(subset) == 0:
        return set()
    return set(subset.iloc[-1]["tickers"].split(","))

# ── Validações cruzadas ───────────────────────────────────────────────────────
validations = [
    ("TSLA",  True,  "2020-12-21", "Tesla entrou em 21/dez/2020"),
    ("TSLA",  False, "2020-12-20", "Tesla NÃO estava em 20/dez/2020"),
    ("FB",    True,  "2022-06-08", "Facebook ainda era FB em 08/jun/2022"),
    ("META",  True,  "2022-06-09", "Meta a partir de 09/jun/2022"),
    ("LEHMQ", False, "2009-01-01", "Lehman removida após falência em 2008"),
    ("AMZN",  True,  "2023-12-31", "Amazon sempre presente"),
]

all_ok = True
for ticker, should_be_in, date_str, note in validations:
    tickers = get_constituents(date_str)
    present = ticker in tickers
    ok = present == should_be_in
    if not ok:
        all_ok = False
    status = "✅" if ok else "❌"
    print(f"  {status} {ticker:6s} {'IN ' if should_be_in else 'OUT'}  {date_str}  — {note}")

print()
print("Todas as validações passaram ✅" if all_ok else "⚠️  Verificar falhas acima.")


  ✅ TSLA   IN   2020-12-21  — Tesla entrou em 21/dez/2020
  ✅ TSLA   OUT  2020-12-20  — Tesla NÃO estava em 20/dez/2020
  ✅ FB     IN   2022-06-08  — Facebook ainda era FB em 08/jun/2022
  ✅ META   IN   2022-06-09  — Meta a partir de 09/jun/2022
  ✅ LEHMQ  OUT  2009-01-01  — Lehman removida após falência em 2008
  ✅ AMZN   IN   2023-12-31  — Amazon sempre presente

Todas as validações passaram ✅


---
## Etapa 6 — Snapshots semestrais e lista de tickers para coleta

Para cada semestre de **2015/S1** a **2025/S2**, extraímos a composição do
índice no último dia do semestre (30/jun ou 31/dez). Isso define quem estava
"dentro" do índice naquele período — respeitando o critério anti-survivorship bias.

O resultado são dois arquivos:
- `snapshots_semestrais.csv` — composição completa por semestre
- `tickers_a_coletar.csv` — lista de tickers únicos com janela de coleta no Yahoo


In [12]:
# Definir os semestres de 2015/S1 a 2025/S2
semesters = []
for ano in range(2015, 2026):
    for sem in [1, 2]:
        inicio = f"{ano}-01-01" if sem == 1 else f"{ano}-07-01"
        fim    = f"{ano}-06-30" if sem == 1 else f"{ano}-12-31"
        semesters.append({
            "periodo":       f"{ano}/S{sem}",
            "ano":           ano,
            "semestre":      sem,
            "inicio":        inicio,
            "fim":           fim,
            "snapshot_date": fim,   # composição no último dia do semestre
        })

# Gerar snapshots
snap_rows = []
for s in semesters:
    t = get_constituents(s["snapshot_date"])
    snap_rows.append({**s, "n_tickers": len(t), "tickers": ",".join(sorted(t))})

snap_df = pd.DataFrame(snap_rows)
snap_df.to_csv(PATH_SNAPSHOTS, index=False)

print(f"Snapshots gerados: {len(snap_df)} semestres")
print()
print(snap_df[["periodo", "n_tickers"]].to_string(index=False))


Snapshots gerados: 22 semestres

periodo  n_tickers
2015/S1        499
2015/S2        502
2016/S1        505
2016/S2        506
2017/S1        506
2017/S2        505
2018/S1        506
2018/S2        505
2019/S1        505
2019/S2        505
2020/S1        505
2020/S2        505
2021/S1        505
2021/S2        505
2022/S1        503
2022/S2        503
2023/S1        503
2023/S2        503
2024/S1        503
2024/S2        503
2025/S1        503
2025/S2        503


In [13]:
# ── Parâmetro de modo de coleta ───────────────────────────────────────────────
#
# MODO_COLETA = "sp500_only"
#   end_date de cada ticker = último semestre em que esteve no S&P 500.
#   Coleta dados apenas do período em que a empresa era constituinte.
#
# MODO_COLETA = "full_history"
#   end_date de cada ticker = 2025-12-31 para todos (ou a data mais recente
#   disponível se a empresa deixou de existir).
#   Mantém dados mesmo após a saída do índice — útil para pair trading porque
#   uma empresa que saiu do S&P 500 ainda pode ser relevante como par.
#
MODO_COLETA = "full_history"   # ← altere para "sp500_only" se preferir

END_DATE_FULL = "2025-12-31"   # data final quando MODO_COLETA = "full_history"

# ── Construir lista de tickers ────────────────────────────────────────────────
ext_df = snap_df[snap_df["periodo"] != "2015/S1"].copy()

all_tickers = set()
for _, row in ext_df.iterrows():
    all_tickers.update(row["tickers"].split(","))

ticker_rows = []
for ticker in sorted(all_tickers):
    periods_in = [
        row["periodo"]
        for _, row in ext_df.iterrows()
        if ticker in row["tickers"].split(",")
    ]
    first_sem, last_sem = periods_in[0], periods_in[-1]
    f_ano, f_s = int(first_sem[:4]), int(first_sem[-1])
    l_ano, l_s = int(last_sem[:4]),  int(last_sem[-1])

    start_date = f"{f_ano}-01-01" if f_s == 1 else f"{f_ano}-07-01"

    if MODO_COLETA == "full_history":
        end_date = END_DATE_FULL
    else:  # "sp500_only"
        end_date = f"{l_ano}-06-30" if l_s == 1 else f"{l_ano}-12-31"

    ticker_rows.append({
        "ticker":       ticker,
        "first_period": first_sem,
        "last_period":  last_sem,
        "n_periods":    len(periods_in),
        "start_date":   start_date,
        "end_date":     end_date,
        "status":       "pending",
    })

tickers_df = pd.DataFrame(ticker_rows)
tickers_df.to_csv(PATH_TICKERS, index=False)

t_2015s1 = set(snap_df[snap_df["periodo"] == "2015/S1"]["tickers"].iloc[0].split(","))
t_2025s2 = set(snap_df[snap_df["periodo"] == "2025/S2"]["tickers"].iloc[0].split(","))
print(f"Modo de coleta: {MODO_COLETA.upper()}")
print(f"Tickers únicos para coletar: {len(tickers_df)}")
print(f"  Permaneceram no índice (2015→2025): {len(t_2015s1 & t_2025s2)}")
print(f"  Entraram no índice:                 {len(t_2025s2 - t_2015s1)}")
print(f"  Saíram do índice:                   {len(t_2015s1 - t_2025s2)}")
if MODO_COLETA == "full_history":
    saiu = t_2015s1 - t_2025s2
    print(f"  → Esses {len(saiu)} que saíram serão coletados até {END_DATE_FULL}")
print(f"\nArquivo salvo: {PATH_TICKERS.name}")


Modo de coleta: FULL_HISTORY
Tickers únicos para coletar: 725
  Permaneceram no índice (2015→2025): 320
  Entraram no índice:                 183
  Saíram do índice:                   179
  → Esses 179 que saíram serão coletados até 2025-12-31

Arquivo salvo: tickers_a_coletar.csv


---
## Etapa 7 — Validação do tipo de ajuste de preço

Antes de coletar os dados do Yahoo, precisamos confirmar qual tipo de preço
a base do professor usa — para garantir consistência na junção das bases.

### O que descobrimos

A base do professor usa **`Close` sem ajuste de dividendos** (split-adjusted apenas),
**não** o `Adj Close` que inclui dividendos.

**Evidência:** ao comparar os últimos valores da base do professor com o
Yahoo Finance no mesmo período, o `ratio` com `Close` bruto fica ≈ 1.0
para MSFT, KO, JNJ e XOM, enquanto o `ratio` com `Adj Close` diverge
significativamente (1.15×, 1.38×, 1.33×, 1.56×).

### Por que a validação exige a base completa

O script usa `Periods.csv` para mapear linha → data exata, evitando o
desalinhamento causado por feriados da NYSE. A base parcial (4k linhas)
usa `bdate_range` como proxy e acumula ~2-3 dias de erro até o final —
suficiente para inverter a correlação dos retornos diários e gerar falsos
negativos na validação.

### Consequência para a coleta (Etapa 8)

Usar `auto_adjust=False` e a coluna `Close` no yfinance — não `Adj Close`.


In [14]:
# Carregar a base completa para uso na validação de preços (Etapa 7)
# e na validação tripla (Etapa 9).
if PATH_DB_FULL.exists():
    print(f"Carregando base completa: {PATH_DB_FULL.name}")
    db_full = pd.read_csv(PATH_DB_FULL)
    print(f"  Shape: {db_full.shape[0]:,} linhas × {db_full.shape[1]:,} colunas")

    # Mapear datas reais via Periods.csv
    # Reseta o índice a cada semestre para conter o drift de feriados
    periods_v = pd.read_csv(PATH_PERIODS, header=None)
    periods_v.columns = ["dias_sem", "nan1", "nan2", "dias_ano"]
    datas = []
    for i, row in periods_v.iterrows():
        n = int(row["dias_sem"])
        if i == 0:   ini = "1990-07-02"
        else:
            ano = 1990 + (i // 2)
            ini = f"{ano}-01-02" if i % 2 == 1 else f"{ano}-07-02"
        datas.extend(pd.bdate_range(start=ini, periods=n))

    n = min(len(db_full), len(datas))
    db_full = db_full.iloc[:n].copy()
    db_full.index = pd.DatetimeIndex(datas[:n])
    print(f"  Datas: {db_full.index[0].date()} → {db_full.index[-1].date()}")
else:
    db_full = None
    print("⚠️  Base completa não encontrada — validações de preço serão puladas.")
    print(f"   Esperado em: {PATH_DB_FULL}")


Carregando base completa: Pt_formatted.csv
  Shape: 6,426 linhas × 1,100 colunas
  Datas: 1990-07-02 → 2015-12-25


In [15]:
# ── Validação do ajuste de preço (Yahoo Close vs Adj Close vs base) ───────────
# Objetivo: confirmar que a base do professor usa Close SEM ajuste de dividendos.
# Resultado já conhecido pela análise anterior: ratio_raw ≈ 1.0 para MSFT/KO/JNJ/XOM.
# Esta célula re-executa a validação de forma compacta para registro no notebook.

TICKERS_VAL_AJUSTE = ["MSFT", "KO", "JNJ", "XOM"]
N_DIAS             = 20
TOLERANCIA         = 0.01

if db_full is None:
    print("⚠️  db_full não carregado — pule para a Etapa 8.")
else:
    print("Comparando base do professor × Yahoo Close × Yahoo Adj Close")
    print(f"Janela: últimos {N_DIAS} pregões da base | tolerância: {TOLERANCIA*100:.0f}%")
    print()

    for ticker in TICKERS_VAL_AJUSTE:
        if ticker not in db_full.columns:
            print(f"{ticker}: não encontrado na base"); continue

        serie = db_full[ticker].dropna().tail(N_DIAS)
        if len(serie) < 5:
            print(f"{ticker}: dados insuficientes"); continue

        d_ini = serie.index[0].strftime("%Y-%m-%d")
        d_fim = (serie.index[-1] + pd.Timedelta(days=5)).strftime("%Y-%m-%d")

        try:
            raw = yf.download(ticker, start=d_ini, end=d_fim,
                              auto_adjust=False, progress=False)
            if hasattr(raw.columns, "levels"):
                raw.columns = raw.columns.get_level_values(0)
            if raw is None or len(raw) == 0:
                print(f"{ticker}: sem dados no Yahoo"); continue

            idx = serie.index.intersection(raw.index)
            if len(idx) < 3:
                print(f"{ticker}: apenas {len(idx)} datas em comum"); continue

            r_close = (serie.loc[idx] / raw["Close"].loc[idx]).dropna()
            r_adj   = (serie.loc[idx] / raw["Adj Close"].loc[idx]).dropna()

            d_close = abs(r_close.mean() - 1.0)
            d_adj   = abs(r_adj.mean()   - 1.0)
            icon    = "✅" if d_close < TOLERANCIA else "⚠️ "
            melhor  = "Close (sem div)" if d_close < d_adj else "Adj Close (com div)"
            print(f"{icon} {ticker:5s}  ratio_Close={r_close.mean():.4f}±{r_close.std():.4f}"
                  f"  ratio_AdjClose={r_adj.mean():.4f}±{r_adj.std():.4f}"
                  f"  → {melhor}")
        except Exception as e:
            print(f"{ticker}: ERRO — {e}")

    print()
    print("✅ Confirmado: base usa Close SEM dividendos → coleta usa auto_adjust=False")


Comparando base do professor × Yahoo Close × Yahoo Adj Close
Janela: últimos 20 pregões da base | tolerância: 1%

✅ MSFT   ratio_Close=1.0029±0.0176  ratio_AdjClose=1.1480±0.0201  → Close (sem div)
✅ KO     ratio_Close=1.0022±0.0157  ratio_AdjClose=1.3817±0.0216  → Close (sem div)
✅ JNJ    ratio_Close=1.0020±0.0159  ratio_AdjClose=1.3287±0.0211  → Close (sem div)
✅ XOM    ratio_Close=0.9967±0.0298  ratio_AdjClose=1.5549±0.0465  → Close (sem div)

✅ Confirmado: base usa Close SEM dividendos → coleta usa auto_adjust=False


---
## Etapa 8 — Coleta de preços via Yahoo Finance

Para cada ticker em `tickers_a_coletar.csv`, baixamos o preço de fechamento
**sem ajuste de dividendos** (`auto_adjust=False`, coluna `Close`) — consistente
com a metodologia da base do professor.

> **Por que não usar `Adj Close`?**
> O `Adj Close` do Yahoo desconta dividendos retroativamente toda vez que um
> novo dividendo é pago, alterando valores históricos a cada atualização.
> A base do professor usa apenas ajuste por splits — usar `Adj Close` criaria
> uma descontinuidade artificial na junção das bases em 2015/S1.

**Checkpoint de retomada:** o `coleta_report.csv` é salvo após cada ticker.
Se a coleta for interrompida, basta re-executar — os tickers já coletados
são pulados automaticamente.

**Tickers `DELISTED`:** empresas que faliram ou foram adquiridas não têm
histórico no Yahoo. Elas serão listadas no sumário final para busca em
fontes alternativas (Tiingo ou EODHD).


In [16]:
# ── Configurações da coleta ───────────────────────────────────────────────────
DELAY_ENTRE_REQUESTS = 0.3   # segundos entre requests (evita rate limit do Yahoo)
MIN_LINHAS_VALIDAS   = 10    # mínimo de dias para considerar o download válido

# Recarregar lista (respeita o MODO_COLETA definido na Etapa 6)
tickers_df   = pd.read_csv(PATH_TICKERS)

# Checkpoint de retomada
if PATH_REPORT.exists():
    report_df    = pd.read_csv(PATH_REPORT)
    already_done = set(report_df["ticker"].tolist())
    print(f"Retomando: {len(already_done)} já coletados, "
          f"{len(tickers_df) - len(already_done)} restantes.")
else:
    report_df    = pd.DataFrame(columns=["ticker", "status", "n_rows",
                                          "start_date", "end_date", "note"])
    already_done = set()
    print(f"Iniciando coleta de {len(tickers_df)} tickers.")

print(f"Ajuste de preço : Close sem dividendos (auto_adjust=False)")
print(f"Modo de coleta  : {tickers_df['end_date'].unique()[:3]} ...")


Retomando: 725 já coletados, 0 restantes.
Ajuste de preço : Close sem dividendos (auto_adjust=False)
Modo de coleta  : ['2025-12-31'] ...


In [17]:
# ── Loop de coleta ────────────────────────────────────────────────────────────
total = len(tickers_df)

for _, row in tickers_df.iterrows():
    ticker     = row["ticker"]
    start_date = row["start_date"]
    end_date   = row["end_date"]

    if ticker in already_done:
        continue

    n_done = len(already_done) + 1
    print(f"[{n_done:4d}/{total}] {ticker:10s}  {start_date} → {end_date}  ", end="", flush=True)

    try:
        # auto_adjust=False → coluna 'Close' é split-adjusted apenas (sem dividendos)
        # Isso é consistente com a metodologia da base do professor.
        data = yf.download(
            ticker,
            start=start_date,
            end=end_date,
            auto_adjust=False,
            progress=False,
        )

        # Normalizar colunas (yfinance pode retornar MultiIndex)
        if hasattr(data.columns, "levels"):
            data.columns = data.columns.get_level_values(0)

        if data is None or len(data) < MIN_LINHAS_VALIDAS:
            status, note, n_rows = "delisted", "sem dados no Yahoo", 0
            print("DELISTED")
        else:
            # Salvar apenas a coluna Close (sem ajuste de dividendos)
            price_series = data[["Close"]].rename(columns={"Close": ticker})
            price_series.to_csv(DIR_PRICES / f"{ticker}.csv")
            status, note, n_rows = "ok", "", len(data)
            print(f"OK  ({n_rows} dias)")

    except Exception as e:
        status, note, n_rows = "error", str(e)[:120], 0
        print(f"ERRO: {note}")

    # Salvar no relatório imediatamente (garante retomada mesmo com crash)
    new_row = pd.DataFrame([{
        "ticker":     ticker,
        "status":     status,
        "n_rows":     n_rows,
        "start_date": start_date,
        "end_date":   end_date,
        "note":       note,
    }])
    report_df    = pd.concat([report_df, new_row], ignore_index=True)
    report_df.to_csv(PATH_REPORT, index=False)
    already_done.add(ticker)

    time.sleep(DELAY_ENTRE_REQUESTS)

print("\nColeta concluída.")



Coleta concluída.


In [18]:
# ── Sumário final da coleta ───────────────────────────────────────────────────
report_df = pd.read_csv(PATH_REPORT)

contagem = report_df["status"].value_counts()
print("=== SUMÁRIO DA COLETA ===")
print(f"  ✅ OK (dados baixados):    {contagem.get('ok', 0)}")
print(f"  ⚠️  Delisted/sem dados:   {contagem.get('delisted', 0)}")
print(f"  ❌ Erros:                 {contagem.get('error', 0)}")
print(f"  Total:                    {len(report_df)}")
print()

# Listar tickers delisted para busca em fontes alternativas (Tiingo / EODHD)
delisted = report_df[report_df["status"] == "delisted"]
if len(delisted) > 0:
    tickers_df = pd.read_csv(PATH_TICKERS)
    print(f"Tickers delisted ({len(delisted)}) — buscar no Tiingo ou EODHD:")
    for _, r in delisted.iterrows():
        t_info = tickers_df[tickers_df["ticker"] == r["ticker"]]
        if len(t_info) > 0:
            t = t_info.iloc[0]
            print(f"  {r['ticker']:10s}  {t['first_period']} → {t['last_period']}")


=== SUMÁRIO DA COLETA ===
  ✅ OK (dados baixados):    592
  ⚠️  Delisted/sem dados:   133
  ❌ Erros:                 0
  Total:                    725

Tickers delisted (133) — buscar no Tiingo ou EODHD:
  AABA        2015/S2 → 2016/S2
  ABC         2015/S2 → 2023/S1
  ABMD        2018/S1 → 2022/S1
  ADS         2015/S2 → 2019/S2
  ADT         2015/S2 → 2015/S2
  AGN         2015/S2 → 2019/S2
  ALXN        2015/S2 → 2021/S1
  ANSS        2017/S1 → 2025/S1
  ANTM        2015/S2 → 2021/S2
  APC         2015/S2 → 2019/S1
  ARG         2015/S2 → 2015/S2
  ARNC        2015/S2 → 2019/S2
  ATVI        2015/S2 → 2023/S1
  BCR         2015/S2 → 2017/S1
  BF.B        2015/S2 → 2025/S2
  BHGE        2015/S2 → 2019/S1
  BLL         2015/S2 → 2021/S2
  BRCM        2015/S2 → 2015/S2
  BRK.B       2015/S2 → 2025/S2
  BXLT        2015/S2 → 2015/S2
  CA          2015/S2 → 2018/S1
  CAM         2015/S2 → 2015/S2
  CBS         2015/S2 → 2019/S1
  CCE         2015/S2 → 2015/S2
  CDAY        2021/S2 → 2023

---
## Etapa 9 — Validação tripla de preços: Base × Yahoo × Tiingo

Antes de usar o Tiingo para recuperar os tickers delisted, precisamos confirmar
que as três fontes são consistentes entre si para tickers que existem em todas.

### Estratégia

Escolhemos empresas que:
1. Existiam **antes de 2015** (logo têm dados na base do professor)
2. Ainda existem hoje (logo têm dados no Yahoo)
3. Têm dados no Tiingo (confirmado pelo plano gratuito)

Comparamos os preços nas **datas exatas** em que as três fontes se sobrepõem,
usando como referência o índice do `db_full` (datas reais via `Periods.csv`).

### O que esperamos

- `ratio Base/Yahoo ≈ 1.0` com std baixo → mesma série, só pequenas diferenças
- `ratio Base/Tiingo ≈ 1.0` com std baixo → Tiingo é confiável como substituto
- `ratio Yahoo/Tiingo ≈ 1.0` com std baixo → as duas fontes de extensão são coerentes

### Token Tiingo

O token é carregado da variável `TIINGO_TOKEN` definida abaixo.  
Plano gratuito: 500 req/hora, histórico completo incluindo tickers delisted.


In [19]:
# ── Configuração do Tiingo ────────────────────────────────────────────────────
TIINGO_TOKEN = "dadfd331f2cb44969b8f7468006d20ad62b13262"

# Tickers para a validação tripla:
# - Existiam na base do professor (pré-2015)
# - Ainda listados hoje (têm dados no Yahoo)
# - Casos bem conhecidos e estáveis (poucos eventos corporativos no período)
TICKERS_VALIDACAO_TRIPLA = ["MSFT", "JNJ", "KO", "PG", "AAPL"]

# Janela de comparação: últimos 30 pregões da base do professor
# (em torno de dez/2014 – jun/2015, dependendo da base disponível)
N_DIAS_TRIPLA = 30

print("Configuração da validação tripla:")
print(f"  Tickers : {TICKERS_VALIDACAO_TRIPLA}")
print(f"  Janela  : últimos {N_DIAS_TRIPLA} pregões da base do professor")
print(f"  Token   : {TIINGO_TOKEN[:8]}... (ok)")


Configuração da validação tripla:
  Tickers : ['MSFT', 'JNJ', 'KO', 'PG', 'AAPL']
  Janela  : últimos 30 pregões da base do professor
  Token   : dadfd331... (ok)


In [20]:
# ── Funções auxiliares de fetch ───────────────────────────────────────────────
import requests

def fetch_tiingo(ticker, start_date, end_date):
    """Busca preços diários no Tiingo (Close sem ajuste de dividendos)."""
    url = f"https://api.tiingo.com/tiingo/daily/{ticker}/prices"
    params = {
        "startDate": start_date,
        "endDate":   end_date,
        "token":     TIINGO_TOKEN,
        "resampleFreq": "daily",
    }
    r = requests.get(url, params=params, timeout=15)
    if r.status_code != 200:
        raise ValueError(f"Tiingo {r.status_code} para {ticker}: {r.text[:200]}")
    data = r.json()
    if not data:
        raise ValueError(f"Tiingo retornou lista vazia para {ticker}")
    df = pd.DataFrame(data)
    df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None)
    df = df.set_index("date").sort_index()
    # Tiingo retorna 'close' (sem ajuste de dividendos) e 'adjClose' (com)
    # Usar 'close' para ser consistente com a base do professor
    if "close" not in df.columns:
        raise ValueError(f"Tiingo não retornou coluna 'close' para {ticker}")
    return df[["close"]].rename(columns={"close": "Close"})


def fetch_yahoo(ticker, start_date, end_date):
    """Busca preços diários no Yahoo Finance (Close sem ajuste de dividendos)."""
    raw = yf.download(ticker, start=start_date, end=end_date,
                      auto_adjust=False, progress=False)
    if raw is None or len(raw) == 0:
        raise ValueError(f"Yahoo sem dados para {ticker}")
    if hasattr(raw.columns, "levels"):
        raw.columns = raw.columns.get_level_values(0)
    df = raw[["Close"]].copy()
    df.index = pd.DatetimeIndex(df.index).tz_localize(None)
    return df


def get_base_series(ticker, db_full):
    """Extrai a série de preços da base do professor para um ticker."""
    if db_full is None:
        raise ValueError("db_full não foi carregado")
    if ticker not in db_full.columns:
        raise ValueError(f"{ticker} não encontrado na base")
    s = db_full[[ticker]].dropna().rename(columns={ticker: "Close"})
    s.index = pd.DatetimeIndex(s.index).tz_localize(None)
    return s


print("Funções fetch_tiingo(), fetch_yahoo() e get_base_series() definidas.")


Funções fetch_tiingo(), fetch_yahoo() e get_base_series() definidas.


In [21]:
# ── Validação tripla: Base × Yahoo × Tiingo ───────────────────────────────────
import numpy as np

if db_full is None:
    print("⚠️  db_full não carregado — carregue a base completa na Etapa 7 antes.")
else:
    print("=" * 70)
    print("VALIDAÇÃO TRIPLA DE PREÇOS")
    print("Fonte A: Base do professor  |  Fonte B: Yahoo  |  Fonte C: Tiingo")
    print("=" * 70)

    resumo = []

    for ticker in TICKERS_VALIDACAO_TRIPLA:
        print(f"\n{'─'*70}")
        print(f"Ticker: {ticker}")

        try:
            # ── Série da base do professor (últimos N dias) ───────────────────
            base = get_base_series(ticker, db_full).tail(N_DIAS_TRIPLA)
            if len(base) < 5:
                print(f"  ⚠️  Dados insuficientes na base ({len(base)} linhas)"); continue

            d_ini = base.index[0].strftime("%Y-%m-%d")
            d_fim = (base.index[-1] + pd.Timedelta(days=5)).strftime("%Y-%m-%d")
            print(f"  Janela base: {d_ini} → {base.index[-1].date()} ({len(base)} pregões)")

            # ── Yahoo ─────────────────────────────────────────────────────────
            try:
                yahoo = fetch_yahoo(ticker, d_ini, d_fim)
            except Exception as e:
                print(f"  ❌ Yahoo falhou: {e}"); yahoo = None

            # ── Tiingo ────────────────────────────────────────────────────────
            try:
                tiingo = fetch_tiingo(ticker, d_ini, d_fim)
            except Exception as e:
                print(f"  ❌ Tiingo falhou: {e}"); tiingo = None

            if yahoo is None and tiingo is None:
                print("  Sem dados de Yahoo nem Tiingo para comparar."); continue

            # ── Interseção de datas ───────────────────────────────────────────
            idx = base.index
            if yahoo  is not None: idx = idx.intersection(yahoo.index)
            if tiingo is not None: idx = idx.intersection(tiingo.index)

            if len(idx) == 0:
                print("  ⚠️  Nenhuma data em comum entre as três fontes."); continue

            print(f"  Datas em comum: {len(idx)} pregões  "
                  f"({idx[0].date()} → {idx[-1].date()})")

            base_p   = base.loc[idx,   "Close"]

            # ── Comparações ───────────────────────────────────────────────────
            def ratio_stats(a, b, label):
                r = (a / b).replace([np.inf, -np.inf], np.nan).dropna()
                if len(r) == 0:
                    print(f"  {label}: sem dados para ratio"); return None
                ok = abs(r.mean() - 1.0) < 0.01 and r.std() < 0.01
                icon = "✅" if ok else "⚠️ "
                print(f"  {icon} {label:25s}  "
                      f"ratio={r.mean():.5f} ± {r.std():.5f}  "
                      f"max_abs_diff={( (a-b).abs().max() ):.4f}")
                return {"label": label, "mean": r.mean(), "std": r.std(),
                        "max_diff": (a - b).abs().max(), "ok": ok}

            stats = []
            if yahoo is not None:
                y = yahoo.loc[idx, "Close"]
                stats.append(ratio_stats(base_p, y, "Base / Yahoo"))
                if tiingo is not None:
                    t = tiingo.loc[idx, "Close"]
                    stats.append(ratio_stats(base_p, t, "Base / Tiingo"))
                    stats.append(ratio_stats(y, t,      "Yahoo / Tiingo"))
            elif tiingo is not None:
                t = tiingo.loc[idx, "Close"]
                stats.append(ratio_stats(base_p, t, "Base / Tiingo"))

            # ── Amostra de valores absolutos ──────────────────────────────────
            print(f"\n  Amostra (últimos 5 dias em comum):")
            cols = {"Base": base_p.tail(5)}
            if yahoo  is not None: cols["Yahoo"]  = yahoo.loc[idx,  "Close"].tail(5)
            if tiingo is not None: cols["Tiingo"] = tiingo.loc[idx, "Close"].tail(5)
            print(pd.DataFrame(cols).round(4).to_string())

            all_ok = all(s["ok"] for s in stats if s is not None)
            resumo.append({"ticker": ticker, "n_comum": len(idx), "ok": all_ok})

        except Exception as e:
            print(f"  ❌ Erro inesperado: {e}")
            resumo.append({"ticker": ticker, "n_comum": 0, "ok": False})

    # ── Sumário ───────────────────────────────────────────────────────────────
    print(f"\n{'='*70}")
    print("SUMÁRIO FINAL")
    print(f"{'='*70}")
    for r in resumo:
        icon = "✅" if r["ok"] else "⚠️ "
        print(f"  {icon} {r['ticker']:6s}  {r['n_comum']:3d} datas em comum  "
              f"{'PASSOU' if r['ok'] else 'DIVERGÊNCIA — verificar'}")

    n_ok = sum(1 for r in resumo if r["ok"])
    print()
    if n_ok == len(resumo):
        print("✅ Todas as comparações passaram.")
        print("   → Tiingo é confiável como fonte para tickers delisted no Yahoo.")
        print("   → Pode prosseguir com a coleta via Tiingo (Etapa 10).")
    else:
        print(f"⚠️  {len(resumo)-n_ok} ticker(s) com divergência.")
        print("   → Verifique os tickers marcados antes de usar o Tiingo.")


VALIDAÇÃO TRIPLA DE PREÇOS
Fonte A: Base do professor  |  Fonte B: Yahoo  |  Fonte C: Tiingo

──────────────────────────────────────────────────────────────────────
Ticker: MSFT
  Janela base: 2015-11-16 → 2015-12-25 (30 pregões)
  Datas em comum: 28 pregões  (2015-11-16 → 2015-12-24)
  ⚠️  Base / Yahoo               ratio=1.00296 ± 0.01575  max_abs_diff=2.0000
  ⚠️  Base / Tiingo              ratio=1.00296 ± 0.01574  max_abs_diff=2.0000
  ✅ Yahoo / Tiingo             ratio=1.00000 ± 0.00002  max_abs_diff=0.0050

  Amostra (últimos 5 dias em comum):
             Base  Yahoo  Tiingo
2015-12-18  55.35  54.13   54.13
2015-12-21  55.82  54.83   54.83
2015-12-22  55.67  55.35   55.35
2015-12-23  55.95  55.82   55.82
2015-12-24  56.55  55.67   55.67

──────────────────────────────────────────────────────────────────────
Ticker: JNJ
  Janela base: 2015-11-16 → 2015-12-25 (30 pregões)
  Datas em comum: 28 pregões  (2015-11-16 → 2015-12-24)
  ⚠️  Base / Yahoo               ratio=1.00097 ± 0.013

---
## Etapa 10 — Coletar tickers delisted via Tiingo

Os 133 tickers marcados como `delisted` na Etapa 8 se dividem em dois grupos:

### Grupo A — Retry no Yahoo com ticker corrigido (12 tickers)
Tickers cujo problema era de **formato ou renomeação** — a empresa existe,
só o ticker mudou. Exemplos:
- `BF.B` → `BF-B` (ponto vira hífen no Yahoo)
- `FB` → `META` (renomeação em jun/2022)
- `ANTM` → `ELV`, `CTL` → `LUMN`, etc.

Estratégia: baixar com o ticker antigo para o período anterior à mudança,
e com o novo ticker para o período posterior. Concatenar as duas séries.

### Grupo B — Tiingo (121 tickers)
Tickers adquiridos, fundidos, falidos ou privatizados — o Yahoo não mantém
o histórico. O Tiingo preserva dados históricos de tickers delistados.

Estratégia: request para cada ticker com o `start_date` e `end_date`
do `coleta_report.csv` (janela exata em que esteve no índice).

### Checkpoint de retomada
O `delisted_report.csv` é salvo após cada ticker — pode interromper e
retomar a qualquer momento.


In [22]:
# ── Mapa de renomeações conhecidas ────────────────────────────────────────────
# ticker_antigo → (ticker_novo, data_de_corte)
# data_de_corte: a partir dessa data o ticker novo passa a existir no Yahoo
RENAME_MAP = {
    "BF.B":  ("BF-B",  "2015-07-01"),  # formato: ponto → hífen (sempre foi assim)
    "BRK.B": ("BRK-B", "2015-07-01"),  # formato: ponto → hífen (sempre foi assim)
    "ANTM":  ("ELV",   "2022-06-28"),  # Anthem → Elevance Health
    "CTL":   ("LUMN",  "2020-09-14"),  # CenturyLink → Lumen Technologies
    "DISCA": ("WBD",   "2022-04-11"),  # Discovery → Warner Bros. Discovery
    "DISCK": ("WBD",   "2022-04-11"),  # Discovery class K → Warner Bros. Discovery
    "FB":    ("META",  "2022-06-09"),  # Facebook → Meta Platforms
    "HCP":   ("PEAK",  "2019-02-05"),  # HCP → Healthpeak Properties
    "KORS":  ("CPRI",  "2017-12-31"),  # Michael Kors → Capri Holdings
    "PKI":   ("RVTY",  "2023-03-06"),  # PerkinElmer → Revvity
    "VIAC":  ("PARA",  "2022-02-16"),  # ViacomCBS → Paramount Global
    "VIAB":  ("PARA",  "2022-02-16"),  # Viacom → Paramount Global
}

# Carregar relatório da coleta Yahoo
report_yahoo = pd.read_csv(PATH_REPORT)
delisted_df  = report_yahoo[report_yahoo["status"] == "delisted"].copy()

grupo_a = delisted_df[delisted_df["ticker"].isin(RENAME_MAP)].copy()
grupo_b = delisted_df[~delisted_df["ticker"].isin(RENAME_MAP)].copy()

print(f"Tickers delisted no Yahoo:     {len(delisted_df)}")
print(f"  Grupo A — retry Yahoo:       {len(grupo_a)} (ticker renomeado/formato)")
print(f"  Grupo B — Tiingo:            {len(grupo_b)} (adquiridas/falidas/privadas)")
print()
print("Grupo A (renomeações):")
for _, r in grupo_a.iterrows():
    novo, corte = RENAME_MAP[r["ticker"]]
    print(f"  {r['ticker']:8s} → {novo:8s}  (corte: {corte})")


Tickers delisted no Yahoo:     133
  Grupo A — retry Yahoo:       12 (ticker renomeado/formato)
  Grupo B — Tiingo:            121 (adquiridas/falidas/privadas)

Grupo A (renomeações):
  ANTM     → ELV       (corte: 2022-06-28)
  BF.B     → BF-B      (corte: 2015-07-01)
  BRK.B    → BRK-B     (corte: 2015-07-01)
  CTL      → LUMN      (corte: 2020-09-14)
  DISCA    → WBD       (corte: 2022-04-11)
  DISCK    → WBD       (corte: 2022-04-11)
  FB       → META      (corte: 2022-06-09)
  HCP      → PEAK      (corte: 2019-02-05)
  KORS     → CPRI      (corte: 2017-12-31)
  PKI      → RVTY      (corte: 2023-03-06)
  VIAB     → PARA      (corte: 2022-02-16)
  VIAC     → PARA      (corte: 2022-02-16)


In [23]:
# ── Checkpoint: carregar progresso anterior se existir ────────────────────────
DELAY_TIINGO = 0.5   # segundos entre requests Tiingo (plano gratuito: 500/hora)

if PATH_DELISTED_REPORT.exists():
    del_report    = pd.read_csv(PATH_DELISTED_REPORT)
    already_del   = set(del_report["ticker"].tolist())
    print(f"Retomando: {len(already_del)} já processados, "
          f"{len(delisted_df) - len(already_del)} restantes.")
else:
    del_report  = pd.DataFrame(columns=["ticker", "fonte", "status",
                                         "n_rows", "start_date", "end_date", "note"])
    already_del = set()
    print(f"Iniciando coleta de {len(delisted_df)} tickers delisted.")


Retomando: 133 já processados, 0 restantes.


In [24]:
# ── GRUPO A: retry Yahoo com ticker corrigido/novo ────────────────────────────
print("=" * 60)
print("GRUPO A — Retry Yahoo com ticker corrigido")
print("=" * 60)

for _, row in grupo_a.iterrows():
    ticker_old = row["ticker"]
    if ticker_old in already_del:
        continue

    ticker_new, data_corte = RENAME_MAP[ticker_old]
    start      = row["start_date"]
    end        = row["end_date"]

    print(f"{ticker_old} → {ticker_new}  ({start} → {end})")

    try:
        frames = []

        # Período 1: antes da renomeação, usar ticker antigo
        # (só se o ticker antigo for diferente por formato, não por nome)
        # Para tickers de formato (BF.B→BF-B), o ticker "antigo" no Yahoo é já o novo
        if ticker_old in ("BF.B", "BRK.B"):
            # Só um ticker a buscar (apenas formato mudou)
            raw = yf.download(ticker_new, start=start, end=end,
                              auto_adjust=False, progress=False)
            if hasattr(raw.columns, "levels"):
                raw.columns = raw.columns.get_level_values(0)
            if raw is not None and len(raw) > 0:
                frames.append(raw[["Close"]].rename(columns={"Close": ticker_old}))
        else:
            # Período antes da renomeação: ticker antigo pode não existir no Yahoo
            # mas tentamos — se falhar, apenas usamos o novo
            if pd.to_datetime(start) < pd.to_datetime(data_corte):
                try:
                    raw_old = yf.download(ticker_old, start=start, end=data_corte,
                                          auto_adjust=False, progress=False)
                    if hasattr(raw_old.columns, "levels"):
                        raw_old.columns = raw_old.columns.get_level_values(0)
                    if raw_old is not None and len(raw_old) > 0:
                        frames.append(raw_old[["Close"]].rename(
                            columns={"Close": ticker_old}))
                        print(f"    Período antigo ({ticker_old}): {len(raw_old)} dias")
                except Exception:
                    pass  # ticker antigo não existe mais no Yahoo — normal

            # Período após a renomeação: ticker novo
            start_new = max(start, data_corte)
            if pd.to_datetime(start_new) <= pd.to_datetime(end):
                raw_new = yf.download(ticker_new, start=start_new, end=end,
                                      auto_adjust=False, progress=False)
                if hasattr(raw_new.columns, "levels"):
                    raw_new.columns = raw_new.columns.get_level_values(0)
                if raw_new is not None and len(raw_new) > 0:
                    frames.append(raw_new[["Close"]].rename(
                        columns={"Close": ticker_old}))
                    print(f"    Período novo ({ticker_new}): {len(raw_new)} dias")

        if frames:
            combined = pd.concat(frames).sort_index()
            # Remover duplicatas de datas (pode ocorrer na data de corte)
            combined = combined[~combined.index.duplicated(keep="first")]
            combined.to_csv(DIR_PRICES / f"{ticker_old}.csv")
            status, note, n_rows = "ok_rename", f"via {ticker_new}", len(combined)
            print(f"    ✅ Total: {n_rows} dias → salvo em prices/{ticker_old}.csv")
        else:
            status, note, n_rows = "delisted", "sem dados mesmo com ticker novo", 0
            print(f"    ❌ Sem dados mesmo com ticker {ticker_new}")

    except Exception as e:
        status, note, n_rows = "error", str(e)[:120], 0
        print(f"    ❌ ERRO: {note}")

    new_row = pd.DataFrame([{"ticker": ticker_old, "fonte": "yahoo_rename",
                              "status": status, "n_rows": n_rows,
                              "start_date": start, "end_date": end, "note": note}])
    del_report  = pd.concat([del_report, new_row], ignore_index=True)
    del_report.to_csv(PATH_DELISTED_REPORT, index=False)
    already_del.add(ticker_old)
    time.sleep(0.3)

print("Grupo A concluído.")


GRUPO A — Retry Yahoo com ticker corrigido
Grupo A concluído.


In [25]:
# ── GRUPO B: Tiingo para tickers adquiridos/falidos/privados ──────────────────
print("=" * 60)
print("GRUPO B — Tiingo (adquiridas, falidas, privadas)")
print("=" * 60)
print(f"Total: {len(grupo_b)} tickers  |  delay: {DELAY_TIINGO}s entre requests")
print()

total_b = len(grupo_b)
for i, (_, row) in enumerate(grupo_b.iterrows(), 1):
    ticker = row["ticker"]
    if ticker in already_del:
        continue

    start = row["start_date"]
    end   = row["end_date"]

    print(f"[{i:3d}/{total_b}] {ticker:10s}  {start} → {end}  ", end="", flush=True)

    try:
        tiingo_df = fetch_tiingo(ticker, start, end)

        if tiingo_df is None or len(tiingo_df) == 0:
            status, note, n_rows = "delisted", "sem dados no Tiingo", 0
            print("SEM DADOS")
        else:
            # Salvar com o nome do ticker original para compatibilidade
            tiingo_df.rename(columns={"Close": ticker}).to_csv(
                DIR_PRICES_TIINGO / f"{ticker}.csv")
            status, note, n_rows = "ok_tiingo", "", len(tiingo_df)
            print(f"OK  ({n_rows} dias)")

    except Exception as e:
        note = str(e)[:120]
        # Tiingo retorna 404 para tickers que não conhece
        if "404" in note or "vazia" in note.lower():
            status, n_rows = "not_found_tiingo", 0
            print(f"NÃO ENCONTRADO")
        else:
            status, n_rows = "error", 0
            print(f"ERRO: {note}")

    new_row = pd.DataFrame([{"ticker": ticker, "fonte": "tiingo",
                              "status": status, "n_rows": n_rows,
                              "start_date": start, "end_date": end, "note": note}])
    del_report  = pd.concat([del_report, new_row], ignore_index=True)
    del_report.to_csv(PATH_DELISTED_REPORT, index=False)
    already_del.add(ticker)
    time.sleep(DELAY_TIINGO)

print("Grupo B concluído.")


GRUPO B — Tiingo (adquiridas, falidas, privadas)
Total: 121 tickers  |  delay: 0.5s entre requests

Grupo B concluído.


In [26]:
# ── Sumário da coleta de delisted ─────────────────────────────────────────────
del_report = pd.read_csv(PATH_DELISTED_REPORT)
contagem   = del_report["status"].value_counts()

print("=== SUMÁRIO ETAPA 10 ===")
print(f"  ✅ ok_rename  (Yahoo, ticker corrigido):  {contagem.get('ok_rename', 0)}")
print(f"  ✅ ok_tiingo  (Tiingo):                   {contagem.get('ok_tiingo', 0)}")
print(f"  ❌ not_found  (nem Yahoo nem Tiingo):      {contagem.get('not_found_tiingo', 0)}")
print(f"  ❌ delisted   (sem dados em nenhuma fonte):{contagem.get('delisted', 0)}")
print(f"  ❌ error      :                            {contagem.get('error', 0)}")
print(f"  Total processados: {len(del_report)}")
print()

# Tickers que não foram encontrados em nenhuma fonte — tratamento para a base final:
# serão preenchidos com NaN (preservamos a coluna mas sem dados)
sem_dados = del_report[del_report["status"].isin(["not_found_tiingo", "delisted", "error"])]
if len(sem_dados) > 0:
    print(f"Tickers sem dados em nenhuma fonte ({len(sem_dados)}):")
    print("  → Colunas serão criadas com NaN na base final (não afeta pair trading)")
    for _, r in sem_dados.iterrows():
        print(f"  {r['ticker']:10s}  {r['start_date']} → {r['end_date']}  ({r['status']})")


=== SUMÁRIO ETAPA 10 ===
  ✅ ok_rename  (Yahoo, ticker corrigido):  3
  ✅ ok_tiingo  (Tiingo):                   82
  ❌ not_found  (nem Yahoo nem Tiingo):      39
  ❌ delisted   (sem dados em nenhuma fonte):9
  ❌ error      :                            0
  Total processados: 133

Tickers sem dados em nenhuma fonte (48):
  → Colunas serão criadas com NaN na base final (não afeta pair trading)
  ANTM        2015-07-01 → 2021-12-31  (delisted)
  CTL         2015-07-01 → 2020-06-30  (delisted)
  DISCA       2015-07-01 → 2021-12-31  (delisted)
  DISCK       2015-07-01 → 2021-12-31  (delisted)
  FB          2015-07-01 → 2021-12-31  (delisted)
  HCP         2015-07-01 → 2019-06-30  (delisted)
  PKI         2015-07-01 → 2022-12-31  (delisted)
  VIAB        2015-07-01 → 2019-06-30  (delisted)
  VIAC        2019-07-01 → 2021-12-31  (delisted)
  ABC         2015-07-01 → 2023-06-30  (not_found_tiingo)
  ADS         2015-07-01 → 2019-12-31  (not_found_tiingo)
  ADT         2015-07-01 → 2015-12-31  

---
## Etapa 10B — Recuperar tickers ainda faltantes

Da Etapa 10 restaram 48 tickers sem dados. Esta etapa tenta recuperar o
máximo possível antes da montagem da base final.

### Grupos de recuperação

**Grupo R1 — Rename falhou (9 tickers):** `FB`, `ANTM`, `CTL`, `DISCA`,
`DISCK`, `HCP`, `PKI`, `VIAB`, `VIAC`. O Grupo A da Etapa 10 tentou o
ticker novo no Yahoo mas falhou. Aqui baixamos diretamente do Tiingo
usando o ticker antigo — o Tiingo preserva histórico de tickers delistados.

**Grupo R2 — Empresas ativas com Tiingo 404 (5 tickers):** `GPS`, `MMC`,
`PEAK`, `RE`, `BLL`. Estas empresas ainda existem na bolsa; o Tiingo
retornou 404 por alguma inconsistência. Aqui tentamos o Yahoo diretamente.

**Grupo R3 — Successores conhecidos (14 tickers):** empresas que foram
absorvidas mas cujos dados vivem sob o ticker do sucessor ou do adquirente.
Ex: `ABC → COR`, `STI → TFC`, `WLTW → WTW`, etc.
Baixamos o período da empresa original usando o ticker sucessor.


In [27]:
# ── Configuração da recuperação ────────────────────────────────────────────────
import requests

# Carregar relatórios existentes para saber o que ainda falta
del_rep  = pd.read_csv(PATH_DELISTED_REPORT)
still_missing = del_rep[~del_rep["status"].isin(["ok_tiingo", "ok_rename"])].copy()
print(f"Tickers ainda sem dados: {len(still_missing)}")
print(still_missing["status"].value_counts().to_string())
print()

# Checkpoint
if PATH_RECOVERY_REPORT.exists():
    rec_report   = pd.read_csv(PATH_RECOVERY_REPORT)
    already_rec  = set(rec_report["ticker"].tolist())
    print(f"Retomando: {len(already_rec)} já processados nesta etapa.")
else:
    rec_report  = pd.DataFrame(columns=["ticker", "grupo", "fonte_usada",
                                         "status", "n_rows", "start_date",
                                         "end_date", "note"])
    already_rec = set()

# ── Grupo R1: rename falhou → tentar Tiingo com ticker ANTIGO ────────────────
R1_TICKERS = ["FB", "ANTM", "CTL", "DISCA", "DISCK", "HCP", "PKI", "VIAB", "VIAC"]

print("=" * 55)
print("GRUPO R1 — Tiingo com ticker original (rename havia falhado)")
print("=" * 55)
for ticker in R1_TICKERS:
    if ticker in already_rec:
        continue
    row = del_rep[del_rep["ticker"] == ticker].iloc[0]
    start, end = row["start_date"], row["end_date"]
    print(f"  {ticker:8s}  {start} → {end}  ", end="", flush=True)
    try:
        tiingo_df = fetch_tiingo(ticker, start, end)
        if tiingo_df is not None and len(tiingo_df) > 0:
            tiingo_df.rename(columns={"Close": ticker}).to_csv(
                DIR_PRICES_TIINGO / f"{ticker}.csv")
            status, n_rows, note = "ok_tiingo_r1", len(tiingo_df), ""
            print(f"OK ({n_rows} dias)")
        else:
            status, n_rows, note = "not_found", 0, "Tiingo empty"
            print("SEM DADOS")
    except Exception as e:
        status, n_rows, note = "error", 0, str(e)[:80]
        print(f"ERRO: {note}")

    new_row = pd.DataFrame([{"ticker": ticker, "grupo": "R1",
                              "fonte_usada": "tiingo_old_ticker",
                              "status": status, "n_rows": n_rows,
                              "start_date": start, "end_date": end, "note": note}])
    rec_report = pd.concat([rec_report, new_row], ignore_index=True)
    rec_report.to_csv(PATH_RECOVERY_REPORT, index=False)
    already_rec.add(ticker)
    time.sleep(0.5)

# ── Grupo R2: ativos com Tiingo 404 → tentar Yahoo diretamente ───────────────
R2_TICKERS = ["GPS", "MMC", "PEAK", "RE", "BLL"]

print()
print("=" * 55)
print("GRUPO R2 — Yahoo direto (empresas ativas com Tiingo 404)")
print("=" * 55)
for ticker in R2_TICKERS:
    if ticker in already_rec:
        continue
    row = del_rep[del_rep["ticker"] == ticker]
    if len(row) == 0:
        row = still_missing[still_missing["ticker"] == ticker]
    if len(row) == 0:
        print(f"  {ticker}: não encontrado nos relatórios"); continue
    row = row.iloc[0]
    start, end = row["start_date"], row["end_date"]
    print(f"  {ticker:8s}  {start} → {end}  ", end="", flush=True)
    try:
        data = yf.download(ticker, start=start, end=end,
                           auto_adjust=False, progress=False)
        if hasattr(data.columns, "levels"):
            data.columns = data.columns.get_level_values(0)
        if data is not None and len(data) >= 10:
            data[["Close"]].rename(columns={"Close": ticker}).to_csv(
                DIR_PRICES / f"{ticker}.csv")
            status, n_rows, note = "ok_yahoo_r2", len(data), ""
            print(f"OK ({n_rows} dias)")
        else:
            status, n_rows, note = "not_found", 0, "Yahoo empty"
            print("SEM DADOS")
    except Exception as e:
        status, n_rows, note = "error", 0, str(e)[:80]
        print(f"ERRO: {note}")

    new_row = pd.DataFrame([{"ticker": ticker, "grupo": "R2",
                              "fonte_usada": "yahoo_direct",
                              "status": status, "n_rows": n_rows,
                              "start_date": start, "end_date": end, "note": note}])
    rec_report = pd.concat([rec_report, new_row], ignore_index=True)
    rec_report.to_csv(PATH_RECOVERY_REPORT, index=False)
    already_rec.add(ticker)
    time.sleep(0.3)

# ── Grupo R3: ticker sucessor ─────────────────────────────────────────────────
# successor = (ticker_novo, data_inicio_sucessor)
# baixamos com o ticker novo para o período em que o ticker antigo estava no índice
# porque o Yahoo/Tiingo só conhece o ticker novo

R3_MAP = {
    # ticker_antigo: (ticker_sucessor, start_override)
    # start_override=None → usar o start_date do report original
    "ABC":  ("COR",   None),        # AmerisourceBergen → Cencora
    "ADS":  ("BFH",   None),        # Alliance Data → Bread Financial
    "CBS":  ("PARA",  None),        # CBS → ViacomCBS → Paramount
    "COG":  ("CTRA",  None),        # Cabot Oil&Gas → Coterra Energy
    "FBHS": ("MBC",   "2021-12-15"),# Fortune Brands → MasterBrand (spinoff dec/2021)
    "FLT":  ("CPAY",  None),        # FleetCor → Corpay
    "HFC":  ("MPC",   None),        # HollyFrontier → Marathon Petroleum
    "INFO": ("SPGI",  None),        # IHS Markit → S&P Global
    "LB":   ("BBWI",  "2021-08-02"),# L Brands → Bath & Body Works spinoff
    "STI":  ("TFC",   "2019-12-09"),# SunTrust → Truist Financial
    "TMK":  ("GL",    "2020-08-10"),# Torchmark → Globe Life
    "WLTW": ("WTW",   None),        # Willis Towers Watson
    "WRK":  ("SW",    "2024-07-01"),# WestRock → Smurfit WestRock
    "FTR":  ("FYBR",  "2021-04-01"),# Frontier (post-bankruptcy) → Frontier FYBR
}

print()
print("=" * 55)
print("GRUPO R3 — Ticker sucessor/adquirente")
print("=" * 55)
for ticker_old, (ticker_new, start_override) in R3_MAP.items():
    if ticker_old in already_rec:
        continue
    row = del_rep[del_rep["ticker"] == ticker_old]
    if len(row) == 0:
        row = still_missing[still_missing["ticker"] == ticker_old]
    if len(row) == 0:
        print(f"  {ticker_old}: não encontrado nos relatórios"); continue
    row = row.iloc[0]
    start = start_override if start_override else row["start_date"]
    end   = row["end_date"]

    print(f"  {ticker_old:8s} → {ticker_new:6s}  {start} → {end}  ", end="", flush=True)
    try:
        data = yf.download(ticker_new, start=start, end=end,
                           auto_adjust=False, progress=False)
        if hasattr(data.columns, "levels"):
            data.columns = data.columns.get_level_values(0)
        if data is not None and len(data) >= 10:
            # Save under the ORIGINAL ticker name for merge consistency
            data[["Close"]].rename(columns={"Close": ticker_old}).to_csv(
                DIR_PRICES / f"{ticker_old}.csv")
            status = "ok_successor"
            n_rows = len(data)
            note   = f"via {ticker_new}"
            print(f"OK ({n_rows} dias via {ticker_new})")
        else:
            status, n_rows, note = "not_found", 0, f"{ticker_new} empty"
            print(f"SEM DADOS ({ticker_new})")
    except Exception as e:
        status, n_rows, note = "error", 0, str(e)[:80]
        print(f"ERRO: {note}")

    new_row = pd.DataFrame([{"ticker": ticker_old, "grupo": "R3",
                              "fonte_usada": f"yahoo_{ticker_new}",
                              "status": status, "n_rows": n_rows,
                              "start_date": start, "end_date": end, "note": note}])
    rec_report = pd.concat([rec_report, new_row], ignore_index=True)
    rec_report.to_csv(PATH_RECOVERY_REPORT, index=False)
    already_rec.add(ticker_old)
    time.sleep(0.3)

# ── Sumário ───────────────────────────────────────────────────────────────────
rec_report = pd.read_csv(PATH_RECOVERY_REPORT)
ok_rec = rec_report[rec_report["status"].str.startswith("ok")]
fail_rec = rec_report[~rec_report["status"].str.startswith("ok")]

print()
print("=== SUMÁRIO ETAPA 10B ===")
print(f"  Recuperados: {len(ok_rec)} / {len(rec_report)}")
print(f"  Ainda faltando: {len(fail_rec)}")
if len(fail_rec) > 0:
    print("  Tickers definitivamente sem dados:")
    for _, r in fail_rec.iterrows():
        print(f"    {r['ticker']:10s} [{r['grupo']}]  {r['note']}")
print()
print("Estes serão incluídos na base final com coluna NaN (sem dados).")
print("Para pair trading isso é aceitável — apenas não serão usados como pares.")


Tickers ainda sem dados: 48
status
not_found_tiingo    39
delisted             9

GRUPO R1 — Tiingo com ticker original (rename havia falhado)
  FB        2015-07-01 → 2021-12-31  ERRO: Tiingo retornou lista vazia para FB
  ANTM      2015-07-01 → 2021-12-31  ERRO: Tiingo retornou lista vazia para ANTM
  CTL       2015-07-01 → 2020-06-30  OK (1259 dias)
  DISCA     2015-07-01 → 2021-12-31  ERRO: Tiingo retornou lista vazia para DISCA
  DISCK     2015-07-01 → 2021-12-31  OK (1639 dias)
  HCP       2015-07-01 → 2019-06-30  ERRO: Tiingo retornou lista vazia para HCP
  PKI       2015-07-01 → 2022-12-31  ERRO: Tiingo retornou lista vazia para PKI
  VIAB      2015-07-01 → 2019-06-30  OK (1006 dias)
  VIAC      2019-07-01 → 2021-12-31  OK (110 dias)

GRUPO R2 — Yahoo direto (empresas ativas com Tiingo 404)
  GPS       2015-07-01 → 2021-12-31  

$GPS: possibly delisted; no timezone found

1 Failed download:
['GPS']: possibly delisted; no timezone found


SEM DADOS
  MMC       2015-07-01 → 2025-12-31  

$MMC: possibly delisted; no timezone found

1 Failed download:
['MMC']: possibly delisted; no timezone found


SEM DADOS
  PEAK      2019-07-01 → 2023-12-31  

...

SEM DADOS (GL)
  WLTW     → WTW     2016-01-01 → 2021-12-31  OK (1510 dias via WTW)
  WRK      → SW      2024-07-01 → 2024-06-30  

$SW: possibly delisted; no price data found  (1d 2024-07-01 -> 2024-06-30) (Yahoo error = "Invalid input - start date cannot be after end date. startDate = 1719806400, endDate = 1719720000")

1 Failed download:
['SW']: possibly delisted; no price data found  (1d 2024-07-01 -> 2024-06-30) (Yahoo error = "Invalid input - start date cannot be after end date. startDate = 1719806400, endDate = 1719720000")


SEM DADOS (SW)
  FTR      → FYBR    2021-04-01 → 2016-12-31  

$FYBR: possibly delisted; no timezone found

1 Failed download:
['FYBR']: possibly delisted; no timezone found


SEM DADOS (FYBR)

=== SUMÁRIO ETAPA 10B ===
  Recuperados: 11 / 28
  Ainda faltando: 17
  Tickers definitivamente sem dados:
    FB         [R1]  Tiingo retornou lista vazia para FB
    ANTM       [R1]  Tiingo retornou lista vazia para ANTM
    DISCA      [R1]  Tiingo retornou lista vazia para DISCA
    HCP        [R1]  Tiingo retornou lista vazia para HCP
    PKI        [R1]  Tiingo retornou lista vazia para PKI
    GPS        [R2]  Yahoo empty
    MMC        [R2]  Yahoo empty
    PEAK       [R2]  Yahoo empty
    RE         [R2]  Yahoo empty
    BLL        [R2]  Yahoo empty
    CBS        [R3]  PARA empty
    FBHS       [R3]  MBC empty
    LB         [R3]  BBWI empty
    STI        [R3]  TFC empty
    TMK        [R3]  GL empty
    WRK        [R3]  SW empty
    FTR        [R3]  FYBR empty

Estes serão incluídos na base final com coluna NaN (sem dados).
Para pair trading isso é aceitável — apenas não serão usados como pares.


---
## Etapa 11 — Unir tudo: base final

### Correção da transição (problema identificado na validação)

O `bdate_range` acumula ~5 dias de deriva ao longo de 25 anos por não
descontar feriados da NYSE. Isso faz as **últimas ~5 linhas da base do
professor** receberem datas de julho de 2015, criando interleaving com
a extensão do Yahoo.

**Solução implementada aqui:**
1. A base do professor é tratada como uma sequência de `N` linhas sem datas
2. A extensão (Yahoo/Tiingo) tem datas reais a partir de 2015-07-01
3. O ponto de junção é definido por **índice de linha**, não por data:
   as primeiras 6426 linhas são do professor, as demais são da extensão
4. Após a junção, geramos um índice de datas sequencial usando a extensão
   como âncora e o Periods.csv para contar para trás

### Nota sobre saltos de escala

Tickers com splits ocorridos **após 2015** terão salto de preço na junção.
Isso é diagnóstico, não um erro — o professor usava splits ajustados até
2015, Yahoo ajusta retroativamente para todos os splits. Para pair trading
(baseado em retornos logarítmicos), este salto não afeta os resultados.


In [33]:
# ── Passo 1: Base do professor como sequência pura (sem datas) ───────────────
print("Passo 1: Preparando base do professor...")

if db_full is None:
    raise RuntimeError("db_full não carregado — execute a Etapa 7 primeiro.")

# Usar db_full SEM o índice de datas (ignoramos as datas do bdate_range)
# A base do professor tem exatamente N linhas que vão de 1990/S2 a 2015/S1
base_prof_raw = pd.read_csv(PATH_DB_FULL)  # ← recarrega sem datas
N_PROF = len(base_prof_raw)
print(f"  Linhas da base do professor: {N_PROF:,}")
print(f"  Tickers: {base_prof_raw.shape[1]:,}")


Passo 1: Preparando base do professor...
  Linhas da base do professor: 6,426
  Tickers: 1,100


In [34]:
# ── Passo 2: Carregar extensão (Yahoo + Tiingo + Recovery) ───────────────────
print("\nPasso 2: Carregando preços da extensão...")

report_y = pd.read_csv(PATH_REPORT)
report_d = pd.read_csv(PATH_DELISTED_REPORT) if PATH_DELISTED_REPORT.exists()            else pd.DataFrame(columns=["ticker","status","n_rows"])
report_r = pd.read_csv(PATH_RECOVERY_REPORT) if PATH_RECOVERY_REPORT.exists()            else pd.DataFrame(columns=["ticker","status","n_rows"])

ok_y   = set(report_y[report_y["status"] == "ok"]["ticker"])
ok_d   = set(report_d[report_d["status"].isin(["ok_tiingo","ok_rename"])]["ticker"])
ok_r1  = set(report_r[report_r["status"] == "ok_tiingo_r1"]["ticker"])
ok_r2  = set(report_r[report_r["status"] == "ok_yahoo_r2"]["ticker"])
ok_r3  = set(report_r[report_r["status"] == "ok_successor"]["ticker"])

print(f"  Yahoo OK:              {len(ok_y)}")
print(f"  Tiingo/rename OK:      {len(ok_d)}")
print(f"  Recovery R1 (Tiingo):  {len(ok_r1)}")
print(f"  Recovery R2 (Yahoo):   {len(ok_r2)}")
print(f"  Recovery R3 (sucess.): {len(ok_r3)}")

ext_frames = []

def load_price_csv(fpath, ticker):
    df = pd.read_csv(fpath, index_col=0, parse_dates=True)
    df.index = pd.DatetimeIndex(df.index).tz_localize(None)
    df.index.name = "date"
    col = ticker if ticker in df.columns else df.columns[0]
    return df[[col]].rename(columns={col: ticker})

# Yahoo prices (DIR_PRICES)
for ticker in sorted(ok_y | ok_r2 | ok_r3):
    fpath = DIR_PRICES / f"{ticker}.csv"
    if fpath.exists():
        try:
            ext_frames.append(load_price_csv(fpath, ticker))
        except Exception as e:
            print(f"  ⚠️  {ticker} (Yahoo): {e}")

# Tiingo prices (DIR_PRICES_TIINGO)
for ticker in sorted(ok_d | ok_r1):
    fpath = DIR_PRICES_TIINGO / f"{ticker}.csv"
    if not fpath.exists():
        fpath = DIR_PRICES / f"{ticker}.csv"  # rename saved to Yahoo dir
    if fpath.exists():
        try:
            ext_frames.append(load_price_csv(fpath, ticker))
        except Exception as e:
            print(f"  ⚠️  {ticker} (Tiingo): {e}")

if not ext_frames:
    raise RuntimeError("Nenhum arquivo de extensão encontrado.")

ext_wide = pd.concat(ext_frames, axis=1).sort_index()
# Keep only the extension period (from 2015-07-01 onwards)
ext_wide = ext_wide[ext_wide.index >= "2015-07-01"]
# Remove rows with ALL NaN (weekends/holidays that slipped in)
ext_wide = ext_wide.dropna(how="all")

print(f"\n  Extensão: {ext_wide.shape[0]:,} linhas × {ext_wide.shape[1]} tickers")
print(f"  Período:  {ext_wide.index[0].date()} → {ext_wide.index[-1].date()}")



Passo 2: Carregando preços da extensão...
  Yahoo OK:              592
  Tiingo/rename OK:      85
  Recovery R1 (Tiingo):  4
  Recovery R2 (Yahoo):   0
  Recovery R3 (sucess.): 7

  Extensão: 2,642 linhas × 688 tickers
  Período:  2015-07-01 → 2025-12-31


In [35]:
# ── Passo 3: Junção correta — por índice, não por data ───────────────────────
print("\nPasso 3: Juntando bases...")

# A extensão tem datas reais. A base do professor tem N linhas que
# terminam em 2015/S1 (junho 2015). O último dia da extensão começa
# em 2015-07-01 (primeiro pregão após o fim da base do professor).
# Usamos isso para gerar datas da base do professor de trás pra frente:
# se a extensão começa em 2015-07-01, o dia anterior é o último dia da base.

EXT_START = ext_wide.index[0]  # 2015-07-01 (ou o primeiro dia real disponível)
print(f"  Extensão começa em: {EXT_START.date()}")

# Gerar calendário NYSE retroativo para a base do professor
# usando o Periods.csv como contagem de dias por semestre
periods_v = pd.read_csv(PATH_PERIODS, header=None)
periods_v.columns = ["dias_sem", "nan1", "nan2", "dias_ano"]

# Gerar datas da extensão para ter o calendário NYSE real
# O truque: Yahoo nos dá os pregões reais de 2015-2025
# Usamos bdate_range apenas para o período da base do professor
# e ancoramos o último dia ao dia ANTERIOR ao início da extensão

# Último pregão antes da extensão (o dia de pregão imediatamente antes de EXT_START)
# Em condições normais = 2015-06-30
import numpy as np

# Contar trading days na extensão que temos
ext_trading_days = ext_wide.index  # esses são datas NYSE reais

# Para a base do professor: usar bdate_range mas com tamanho exato do Periods.csv
# e ancorando ao fim conhecido (jun/2015), gerando de trás pra frente
total_days_prof = int(periods_v["dias_sem"].sum())  # exatamente 6425 dias
print(f"  Total dias professor (Periods.csv): {total_days_prof:,}")

# Gerar sequência de trading days terminando em 2015-06-30
# bdate_range de trás pra frente: usamos um período suficientemente longo
# e pegamos as últimas N datas
end_prof = pd.Timestamp("2015-06-30")
# Generate enough business days back
bdays_back = pd.bdate_range(end="2015-06-30",
                             periods=total_days_prof + 300)  # buffer
# Take the last total_days_prof that end on or before 2015-06-30
bdays_back = bdays_back[bdays_back <= end_prof][-total_days_prof:]

print(f"  Datas geradas para base professor: {len(bdays_back)}")
print(f"  Primeira: {bdays_back[0].date()}  →  Última: {bdays_back[-1].date()}")

# Assign these dates to the professor base
n_use = min(len(base_prof_raw), len(bdays_back))
base_prof_dated = base_prof_raw.iloc[:n_use].copy()
base_prof_dated.index = pd.DatetimeIndex(bdays_back[:n_use])
base_prof_dated.index.name = "date"

# Verify no overlap with extension
last_prof_date = base_prof_dated.index[-1]
first_ext_date = ext_wide.index[0]
gap_days = (first_ext_date - last_prof_date).days
print(f"\n  Último dia professor:  {last_prof_date.date()}")
print(f"  Primeiro dia extensão: {first_ext_date.date()}")
print(f"  Gap entre bases:       {gap_days} dias calendário")
if gap_days == 1 or gap_days == 3:  # 1 = consecutive, 3 = weekend
    print(f"  ✅ Junção limpa — sem sobreposição")
elif gap_days < 0:
    print(f"  ⚠️  Sobreposição de {-gap_days} dias — verificar")
else:
    print(f"  ⚠️  Gap de {gap_days} dias — pode haver pregões perdidos")

# Concatenate
sp500_final = pd.concat([base_prof_dated, ext_wide], axis=0, sort=True)
sp500_final = sp500_final.sort_index()
sp500_final = sp500_final[~sp500_final.index.duplicated(keep="last")]
sp500_final = sp500_final.dropna(how="all")

print(f"\n  Shape final: {sp500_final.shape[0]:,} linhas × {sp500_final.shape[1]:,} colunas")



Passo 3: Juntando bases...
  Extensão começa em: 2015-07-01
  Total dias professor (Periods.csv): 6,425
  Datas geradas para base professor: 6425
  Primeira: 1990-11-14  →  Última: 2015-06-30

  Último dia professor:  2015-06-30
  Primeiro dia extensão: 2015-07-01
  Gap entre bases:       1 dias calendário
  ✅ Junção limpa — sem sobreposição

  Shape final: 9,067 linhas × 1,336 colunas


In [31]:
# ── Passo 4: Validação da transição ──────────────────────────────────────────
print("\nPasso 4: Validando a transição...")
print()

# Test tickers: no splits after 2015 (should have ratio ≈ 1.0)
no_split_after_2015 = ["MSFT", "KO", "JNJ", "PG"]
# Test tickers: splits after 2015 (will show expected ratio jump)
split_after_2015 = {
    "AAPL":  ("4:1 ago/2020",             4.0),
    "AMZN":  ("20:1 jun/2022",           20.0),
    "GOOGL": ("20:1 jul/2022",           20.0),
    "NVDA":  ("4:1 jul/2021 + 10:1 jun/2024", 40.0),
}

print("─── Tickers SEM splits pós-2015 (ratio deve ser ≈ 1.0) ───")
for ticker in no_split_after_2015:
    if ticker not in sp500_final.columns:
        print(f"  {ticker}: não encontrado"); continue
    before = sp500_final.loc[:"2015-06-30", ticker].dropna()
    after  = sp500_final.loc["2015-07-01":, ticker].dropna()
    if len(before) == 0 or len(after) == 0:
        print(f"  {ticker}: dados insuficientes"); continue
    ratio  = after.iloc[0] / before.iloc[-1]
    ok     = abs(ratio - 1.0) < 0.03
    print(f"  {'✅' if ok else '⚠️ '} {ticker:6s}: "
          f"último_prof={before.iloc[-1]:.4f}  "
          f"primeiro_ext={after.iloc[0]:.4f}  "
          f"ratio={ratio:.4f}")

print()
print("─── Tickers COM splits pós-2015 (ratio deve ≈ fator de split) ───")
for ticker, (split_info, expected) in split_after_2015.items():
    if ticker not in sp500_final.columns:
        print(f"  {ticker}: não encontrado"); continue
    before = sp500_final.loc[:"2015-06-30", ticker].dropna()
    after  = sp500_final.loc["2015-07-01":, ticker].dropna()
    if len(before) == 0 or len(after) == 0:
        print(f"  {ticker}: dados insuficientes"); continue
    ratio  = after.iloc[0] / before.iloc[-1]
    ok     = abs(ratio - expected) < expected * 0.05
    print(f"  {'✅' if ok else '⚠️ '} {ticker:6s}: ratio={ratio:.1f}x "
          f"(esperado {expected:.0f}x) | {split_info}")

print()
print("─── Verificação de não-duplicação de datas ───")
dupes = sp500_final.index.duplicated().sum()
print(f"  Datas duplicadas: {dupes}  {'✅' if dupes == 0 else '❌'}")

print()
print("─── Contagem de tickers por origem ───")
# Professor-only tickers (not in extension)
ext_tickers  = set(ext_wide.columns)
prof_tickers = set(base_prof_raw.columns)
only_prof    = prof_tickers - ext_tickers
only_ext     = ext_tickers  - prof_tickers
both         = prof_tickers & ext_tickers
print(f"  Só na base do professor: {len(only_prof)} tickers (pre-2015 delistados)")
print(f"  Em ambas as bases:       {len(both)} tickers")
print(f"  Só na extensão:         {len(only_ext)} tickers (entraram após 2015)")
print(f"  Total na base final:    {sp500_final.shape[1]:,} tickers")



Passo 4: Validando a transição...

─── Tickers SEM splits pós-2015 (ratio deve ser ≈ 1.0) ───
  ⚠️  MSFT  : último_prof=56.3100  primeiro_ext=44.4500  ratio=0.7894
  ⚠️  KO    : último_prof=43.5700  primeiro_ext=39.5000  ratio=0.9066
  ⚠️  JNJ   : último_prof=103.7800  primeiro_ext=98.4700  ratio=0.9488
  ✅ PG    : último_prof=79.3782  primeiro_ext=79.7200  ratio=1.0043

─── Tickers COM splits pós-2015 (ratio deve ≈ fator de split) ───
  ⚠️  AAPL  : ratio=0.3x (esperado 4x) | 4:1 ago/2020
  ⚠️  AMZN  : ratio=0.0x (esperado 20x) | 20:1 jun/2022
  ⚠️  GOOGL : ratio=0.0x (esperado 20x) | 20:1 jul/2022
  ⚠️  NVDA  : ratio=0.0x (esperado 40x) | 4:1 jul/2021 + 10:1 jun/2024

─── Verificação de não-duplicação de datas ───
  Datas duplicadas: 0  ✅

─── Contagem de tickers por origem ───
  Só na base do professor: 648 tickers (pre-2015 delistados)
  Em ambas as bases:       452 tickers
  Só na extensão:         236 tickers (entraram após 2015)
  Total na base final:    1,336 tickers


In [32]:
# ── Passo 5: Salvar base final ────────────────────────────────────────────────
print("\nPasso 5: Salvando base final...")

sp500_final.index.name = "date"
sp500_final.to_csv(PATH_FINAL_DB)

import os
size_mb = os.path.getsize(PATH_FINAL_DB) / (1024**2)

print(f"  ✅ Salvo em: {PATH_FINAL_DB.name}")
print(f"  Tamanho:    {size_mb:.1f} MB")
print(f"  Shape:      {sp500_final.shape[0]:,} linhas × {sp500_final.shape[1]:,} tickers")
print()
print("=" * 60)
print("BASE FINAL CONCLUÍDA")
print("=" * 60)
print(f"  Histórico (professor): {sp500_final.index[0].date()} → 2015-06-30")
print(f"  Extensão (Yahoo+Tiingo): 2015-07-01 → {sp500_final.index[-1].date()}")
print(f"  Total de tickers:  {sp500_final.shape[1]:,}")
print(f"  Total de pregões:  {sp500_final.shape[0]:,}")
print()
print("  ⚠️  Nota sobre escala:")
print("  Tickers com splits ocorridos APÓS 2015 têm descontinuidade")
print("  de preço na junção. Isso é esperado e não afeta pair trading,")
print("  que opera sobre retornos logarítmicos (pct_change ou log).")



Passo 5: Salvando base final...
  ✅ Salvo em: sp500_final.csv
  Tamanho:    60.5 MB
  Shape:      9,067 linhas × 1,336 tickers

BASE FINAL CONCLUÍDA
  Histórico (professor): 1990-11-14 → 2015-06-30
  Extensão (Yahoo+Tiingo): 2015-07-01 → 2025-12-31
  Total de tickers:  1,336
  Total de pregões:  9,067

  ⚠️  Nota sobre escala:
  Tickers com splits ocorridos APÓS 2015 têm descontinuidade
  de preço na junção. Isso é esperado e não afeta pair trading,
  que opera sobre retornos logarítmicos (pct_change ou log).
